In [ ]:
from utils import *
warnings.filterwarnings("ignore")
lag_set_daily = np.array([14, 7, 0])
lag_set_weekly = np.array([2, 1, 0])
state_list = state_abbr

train_period = 280
validation_period = 98
intercept_length = validation_period
test_period = 189
# test_period = 189
step_size = 28

dataset_list = ['JHUcase']
# dataset_list = ['HHSflu']
method_list = ['grpLasso'] # 'derandomKnock', 
Only = ['U07']
for dataset in dataset_list:
    target_data = pd.read_csv('../data/' + dataset + '.txt', index_col = 0)
    target_data = target_data.loc[:, state_list].astype(float)
    target_data = target_data.rolling(window=7, min_periods=1).sum()[::7]
    target_dates = target_data.index
    state_data_dict = {}
    for state in state_list:
        state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).rolling(window=7, min_periods=1).sum().iloc[6:]
        state_data = state_data.drop(Only, axis=1)
        state_data_dict[state] = state_data[state_data.index.isin(target_dates)]
    all_codes = state_data_dict[state_list[0]].columns.to_numpy()
    
    target_test = pd.read_csv('../data/' + dataset + '.txt', index_col = 0)
    target_test = target_test.loc[:, state_list].astype(float)
    target_test = target_test.apply(moving_average_smoother).iloc[6:]
    target_test_dates = target_test.index
    state_test_dict = {}
    for state in state_list:
        state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).apply(moving_average_smoother).iloc[6:]
        state_data = state_data.drop(Only, axis=1)
        state_test_dict[state] = state_data[state_data.index.isin(target_test_dates)]
    
    train_start_date_list = []
    validation_start_date_list = []
    test_start_date_list = []
    train_start_date = date_after(max(min(target_data.index), min(state_data_dict[state].index)), max(lag_set_daily))
    # train_start_date = 20211101
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = validation_start_date
    while test_start_date < 20220901:
    # while test_start_date < 20230401:
        train_start_date_list.append(train_start_date)
        validation_start_date_list.append(validation_start_date)
        test_start_date_list.append(test_start_date)
        train_start_date = date_after(train_start_date, step_size)
        validation_start_date = date_after(train_start_date, train_period)
        test_start_date = validation_start_date

    data_handler = HealthDataHandler(state_list, lag_set_weekly, 7, target_data, state_data_dict)
    test_handler = HealthDataHandler(state_list, lag_set_daily, 1, target_test, state_test_dict)
    for method in method_list:
        phe_cnt_dict = {}
        y_weekly = {}
        y_daily = {}
        pred_dict = {}
        pred_agg_dict = {}
        date_plot_list = []
        date_test_plot_list = []
        
        for state in state_list:
            y_weekly[state] = []
            y_daily[state] = []
            pred_dict[state] = []
            pred_agg_dict[state] = []
        
        states_prev_W = None
        states_prev_g = None
        selection = False
        for d_idx in range(len(train_start_date_list)):
            data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
            data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
            print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
            
            data_handler.run_all_single(intercept_length, method, alpha = 0.2, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
            for state in state_list:
                for g in data_handler.active_g_indices[state]:
                    if all_codes[g] in phe_cnt_dict:
                        phe_cnt_dict[all_codes[g]] += 1
                    else:
                        phe_cnt_dict[all_codes[g]] = 1
                        
            data_handler.run_all_agg(intercept_length, state_method = 'marginal_diff', auxiliary_state_list = None, nStates = 5, total_step = 10, selection = selection)
            
            data_handler.get_test(start_test = test_start_date_list[d_idx], codes = all_codes, period_length = test_period)
            test_handler.get_test(start_test = test_start_date_list[d_idx], codes = all_codes, period_length = test_period)
            date_plot_list.append([datetime.strptime(str(d), '%Y%m%d') for d in data_handler.test_dates])
            date_test_plot_list.append([datetime.strptime(str(d), '%Y%m%d') for d in test_handler.test_dates])
            
            for state in state_list:
                y_weekly[state].append(data_handler.y_test[state])
                y_daily[state].append(test_handler.y_test[state])
                pred_dict[state].append(np.dot(test_handler.grouped_X_test[state], data_handler.recovered_coef[state]) + data_handler.recovered_intercept[state]/7)
                pred_agg_dict[state].append(np.dot(test_handler.grouped_X_test[state], data_handler.recovered_coef_agg[state]) + data_handler.recovered_intercept_agg[state]/7)

            dfout = {
                'date': date_test_plot_list[d_idx],
                **{f'y_{state}': y_daily[state][d_idx] for state in state_list},
                **{f'pred_{state}': pred_dict[state][d_idx] for state in state_list},
                **{f'pred_agg_{state}': pred_agg_dict[state][d_idx] for state in state_list}
            }
            dfout = pd.DataFrame(dfout)
            dfout.to_csv(f'test0217_/{dataset}_{method}_{d_idx}.csv', index=False)
            
        top10 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:10]
        for key in top10:
            print(f"{key}: {phe_cnt_dict[key]}")
        print()
        
        for state in state_list:
            plt.figure(figsize=(10, 6))
            for d_idx in range(len(train_start_date_list)):
                plt.plot(date_plot_list[d_idx], y_weekly[state][d_idx]/7, c = 'k', lw = 2, alpha = 0.5)
                plt.plot(date_test_plot_list[d_idx], y_daily[state][d_idx], c = 'b', lw = 1.5, alpha = 0.4)
                plt.plot(date_test_plot_list[d_idx], pred_dict[state][d_idx], alpha = 0.8, lw = 2)
            plt.plot([], [], c='k', lw=2, alpha=0.5, label='weekly average')
            plt.plot([], [], c='b', lw=2, alpha=0.5, label='daily moving average')
            plt.title(state + ', single')
            plt.legend()
            plt.xticks(rotation = 30)
            plt.figure(figsize=(10, 6))
            for d_idx in range(len(train_start_date_list)):
                plt.plot(date_plot_list[d_idx], y_weekly[state][d_idx]/7, c = 'k', lw = 2, alpha = 0.5)
                plt.plot(date_test_plot_list[d_idx], y_daily[state][d_idx], c = 'b', lw = 1.5, alpha = 0.4)
                plt.plot(date_test_plot_list[d_idx], pred_agg_dict[state][d_idx], alpha = 0.8, lw = 2)
            plt.plot([], [], c='k', lw=2, alpha=0.5, label='weekly average')
            plt.plot([], [], c='b', lw=2, alpha=0.5, label='daily moving average')
            plt.title(state + ', aggregated')
            plt.legend()
            plt.xticks(rotation = 30)
        

In [ ]:
from utils import *
warnings.filterwarnings("ignore")

state_list = ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 
                'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MS', 'MT', 'NC', 'NE', 'NJ', 'NM', 
                'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'TN', 'TX', 'UT', 'VA', 'VT', 'WI', 'WV', 'WY']

lag_set_daily = np.array([14, 7, 0])
lag_set_weekly = np.array([2, 1, 0])
group_size = len(lag_set_weekly)

target_data = pd.read_csv('../data/RSV_visits.txt', index_col = 0)
target_data = target_data.loc[:, state_list].astype(float)
target_dates = target_data.index

state_data_dict = {}
for state in state_list:
    state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).rolling(window=7, min_periods=1).sum().iloc[6:]
    state_data_dict[state] = state_data[state_data.index.isin(target_dates)]
all_codes = state_data_dict[state_list[0]].columns.to_numpy()
state_test_dict = {}
for state in state_list:
    state_data = pd.read_csv(f'../data/{state}states.csv', index_col = 0, header = 0).astype(float).apply(moving_average_smoother).iloc[6:]
    state_test_dict[state] = state_data[(state_data.index >= target_dates[0]) & (state_data.index <= target_dates[-1])]

states_prev_W = None
states_prev_g = None
selection = False

data_handler = HealthDataHandler(state_list, lag_set_weekly, 7, target_data, state_data_dict)
test_handler = HealthDataHandler(state_list, lag_set_daily, 1, target_data, state_test_dict)

start_date = max(min(target_data.index), date_after(min(state_data_dict[state].index), max(lag_set_daily)))
end_date = min(max(target_data.index), max(state_data_dict[state].index))
validation_period = (datetime.strptime(str(end_date), '%Y%m%d') - datetime.strptime(str(start_date), '%Y%m%d')).days
intercept_length = validation_period

data_handler.get_X_y(start_date = start_date, end_date = end_date, codes = all_codes)
data_handler.get_test(start_test = start_date, codes = all_codes, period_length = validation_period)
test_handler.get_test(start_test = start_date, codes = all_codes, period_length = validation_period)

data_handler.run_all_single(intercept_length, 'derandomKnock', alpha = 0.2, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)

for state in state_list:
    print(state, end = ':')
    for g_idx in data_handler.active_g_indices[state]:
        print(all_codes[g_idx], end=' ')
    print()
    
data_handler.run_all_agg(intercept_length, state_method = 'marginal_diff', auxiliary_state_list = None, nStates = 5, total_step = 10, selection = selection)

y_weekly = {}
y_daily = {}
pred_dict = {}
pred_agg_dict = {}
for state in state_list:
    y_weekly[state] = data_handler.y_test[state]
    y_daily[state] = test_handler.y_test[state]
    pred_dict[state] = np.dot(test_handler.grouped_X_test[state], data_handler.recovered_coef[state]) + data_handler.recovered_intercept[state]/7
    pred_agg_dict[state] = np.dot(test_handler.grouped_X_test[state], data_handler.recovered_coef_agg[state]) + data_handler.recovered_intercept_agg[state]/7

date_plot = [datetime.strptime(str(d), '%Y%m%d') for d in data_handler.test_dates]

end_test = date_after(start_date, validation_period)
date_test_plot = test_handler.state_data[state].loc[start_date:end_test - 1, :].index.to_numpy()
date_test_plot = [datetime.strptime(str(d), '%Y%m%d') for d in date_test_plot]

for state in state_list:
    plt.figure(figsize=(10, 6))
    plt.plot(date_plot, y_weekly[state]/7, c = 'k', lw = 2, alpha = 0.5)
    plt.plot(date_test_plot, pred_dict[state], alpha = 0.8, lw = 2, label = 'single')
    plt.plot(date_test_plot, pred_agg_dict[state], alpha = 0.8, lw = 2, label = 'aggregated')
    plt.xticks(rotation = 30)
    plt.legend()
    plt.title(state)

In [ ]:
phe_cnt_dict = {}
for state in state_list:
    for g in data_handler.active_g_indices[state]:
        if all_codes[g] in phe_cnt_dict:
            phe_cnt_dict[all_codes[g]] += 1
        else:
            phe_cnt_dict[all_codes[g]] = 1
for key in sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True):
    print(f"{key}: {phe_cnt_dict[key]}")
print()